# 03. Обучение WGAN-GP на эмбеддингах

Conditional WGAN-GP. Условие — категория товара (10 классов).

**Что делаем:**
1. Загружаем (N, 768) эмбеддинги + (N,) labels
2. DataLoader
3. Тренируем 200 эпох
4. Сохраняем чекпоинт + loss curves
5. Генерируем сэмплы для оценки в notebook 05


In [ ]:
# === Colab setup (skip if running locally) ===
import os, sys

IN_COLAB = "COLAB_GPU" in os.environ
if IN_COLAB:
    # Клонируем репо (первый раз) и переходим в него
    if not os.path.exists("/content/gan-reviews"):
        !git clone https://github.com/USER/gan-reviews.git /content/gan-reviews
    %cd /content/gan-reviews
    !pip install -q -r requirements-trainer.txt

# Добавляем корень репо в sys.path, чтобы импорты работали из notebooks/
sys.path.insert(0, os.path.abspath(".."))

# Mount Google Drive для сохранения чекпоинтов (опционально)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/gan-reviews/checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "../checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("CHECKPOINT_DIR =", CHECKPOINT_DIR)


In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from pathlib import Path

from models.wgan_gp import Generator, Critic, train_wgan_gp, sample

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

EMB_DIR = Path('../data/embeddings')
X = np.load(EMB_DIR / 'X_cls.npy').astype('float32')
y = np.load(EMB_DIR / 'labels.npy')
print('X:', X.shape, 'y:', y.shape, 'classes:', y.max() + 1)


In [ ]:
# Стандартизация (важно для GAN — иначе шум сильно отличается от данных)
mean = X.mean(axis=0, keepdims=True)
std = X.std(axis=0, keepdims=True) + 1e-6
X_norm = (X - mean) / std
np.save(EMB_DIR / 'X_cls_mean.npy', mean)
np.save(EMB_DIR / 'X_cls_std.npy', std)

ds = TensorDataset(torch.from_numpy(X_norm), torch.from_numpy(y))
dl = DataLoader(ds, batch_size=64, shuffle=True, drop_last=True)


In [ ]:
NUM_CLASSES = int(y.max() + 1)
gen = Generator(z_dim=128, emb_dim=768, num_classes=NUM_CLASSES)
crit = Critic(emb_dim=768, num_classes=NUM_CLASSES)

history = train_wgan_gp(
    gen, crit, dl,
    n_epochs=200,
    n_critic=5,
    lambda_gp=10.0,
    lr=1e-4,
    device=device,
    log_every=10,
)


In [ ]:
torch.save({
    'gen': gen.state_dict(),
    'crit': crit.state_dict(),
    'history': history,
    'num_classes': NUM_CLASSES,
}, f'{CHECKPOINT_DIR}/wgan_gp.pth')
print('saved')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['d_loss'], label='D (critic)')
axes[0].plot(history['g_loss'], label='G')
axes[0].legend(); axes[0].set_title('Losses'); axes[0].set_xlabel('epoch')
axes[1].plot(history['gp']); axes[1].set_title('Gradient Penalty')
plt.tight_layout()
plt.savefig('../results/wgan_gp_losses.png', dpi=120)
plt.show()


In [ ]:
# Генерируем сэмплы для оценки
fake, fake_labels = sample(gen, n_per_class=500, num_classes=NUM_CLASSES, device=device)
fake_np = fake.cpu().numpy()
# Обратная стандартизация
fake_denorm = fake_np * std + mean
np.save('../data/embeddings/X_gen_wgan.npy', fake_denorm)
np.save('../data/embeddings/y_gen_wgan.npy', fake_labels.cpu().numpy())
print('saved fake samples:', fake_denorm.shape)
